# Building a RAG Ingestion Pipeline with pgvector

This is **Part A** of a two-part notebook. Here we build the **document ingestion pipeline**
that turns raw PDFs into a searchable vector store:

1. **PDF → Markdown** — convert research papers using [MarkItDown](https://github.com/microsoft/markitdown)
2. **Chunk** — split markdown into retrieval-friendly pieces
3. **Embed & Store** — embed the chunks and persist them in pgvector

```mermaid
graph LR
    PDF["📄 PDFs<br/>(rag_data/)"] -->|MarkItDown| MD["📝 Markdown"]
    MD -->|RecursiveCharacterTextSplitter| Chunks["📦 Chunks"]
    Chunks -->|databricks-gte-large-en| PG["🐘 pgvector"]
```

The output is a persistent pgvector collection (`langgraph_rag_demo`) that
**Part B** (`11b.langgraph_rag.ipynb`) consumes to build a LangGraph RAG agent —
no re-embedding required.

**Source papers (in `rag_data/`):**
- Retrieval-Augmented Generation (Lewis et al., 2020)
- AutoGen: Enabling Next-Gen LLM Apps via Multi-Agent Conversations
- Efficient Memory Management for LLM Serving with PagedAttention
- Transformer Explainer: Interactive Learning of Text-Generative Models

**Prerequisites:**
- pgvector pod running (`podman pod start pgvector-pod`)
- `.env` file configured
- `markitdown` installed (`uv pip install markitdown`)

## Step 1 — Bootstrap & Connect

Instead of repeating setup in every notebook, we run a shared configuration script once. It reads
credentials and endpoints from `.env` and hands back ready-to-use clients.

**What this establishes:**
- **An embeddings client** for `databricks-gte-large-en` (1024-dimensional vectors), pre-configured
  to respect the Databricks batch limit so large ingestions don't fail.
- **A connection to the pgvector database** where the embeddings will be stored.
- **The `PGVector` vector-store interface** used to read and write those embeddings.

**Why it's done this way:**
- Configuration lives in one place, so it's consistent and easy to change.
- Part A (this notebook) and Part B are guaranteed to use the **same** database and embedding
  model — which is what lets them share a vector store without re-embedding.

In [ ]:
%run ../langchain_common.py

## Step 2 — Convert PDFs to Markdown with MarkItDown

[MarkItDown](https://github.com/microsoft/markitdown) turns PDFs into clean Markdown. This matters
because structured Markdown — with its headings, lists, and tables intact — chunks and retrieves
far better than the messy text you get from raw PDF extraction.

**The logic of this stage:**
- **Convert once, reuse forever.** Each PDF is converted to a Markdown file that's kept on disk.
  On later runs the saved Markdown is loaded directly, so the slow conversion doesn't repeat.
- **Only reconvert when needed.** A PDF is re-converted only if its Markdown is missing or older
  than the PDF itself — an easy way to detect that the source changed. A manual override
  (`FORCE_RECONVERT`) forces a full rebuild when you want one.
- **Attach a source label to every document.** Each paper's text is tagged with the file it came
  from. This label follows the content through chunking and embedding, and is ultimately what lets
  answers cite the paper they were drawn from.

**Why it matters:**
- **Caching** keeps reruns fast and cheap.
- **Traceability** — that source tag is the thread connecting a retrieved passage back to its origin.

In [ ]:
import os
import re
from pathlib import Path
from markitdown import MarkItDown
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = Path("rag_data")
MD_CACHE_DIR = DATA_DIR / "markdown"          # cached .md files live here
MD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
md_converter = MarkItDown()


def clean_markdown(text: str) -> str:
    """Strip PDF-extraction noise (e.g. the vertical arXiv margin stamp).

    Side-printed text often comes out as many lines that each hold a single
    character, sometimes separated by blank lines. We buffer such runs (blank
    lines included) and drop any run with several single-character lines, so
    chunks contain real prose instead of one-letter-per-line garbage.
    """
    lines = text.splitlines()
    cleaned: list[str] = []
    run: list[str] = []      # buffered single-char + interspersed blank lines
    single_count = 0         # how many buffered lines are single characters

    def flush_run() -> None:
        nonlocal single_count
        # A run with several 1-char lines is margin/vertical text — discard it.
        if single_count < 4:
            cleaned.extend(run)
        run.clear()
        single_count = 0

    for line in lines:
        stripped = line.strip()
        if len(stripped) == 1:
            run.append(line)
            single_count += 1
        elif stripped == "":
            if run:
                run.append(line)      # keep buffering blanks inside a run
            else:
                cleaned.append(line)
        else:
            flush_run()
            cleaned.append(line)
    flush_run()

    out = "\n".join(cleaned)
    out = re.sub(r"\n{3,}", "\n\n", out)  # collapse excessive blank lines
    return out.strip()


# ── Step 2a: Convert PDFs to Markdown (with on-disk cache) ─────────────────────
# On first run each PDF is converted and its markdown is saved next to a cache
# folder. On later runs we load the cached .md instead of re-converting, unless
# the PDF is newer than the cache (or FORCE_RECONVERT is set).
FORCE_RECONVERT = False

docs = []
for pdf_path in sorted(DATA_DIR.glob("*.pdf")):
    md_path = MD_CACHE_DIR / f"{pdf_path.stem}.md"
    cache_fresh = md_path.exists() and md_path.stat().st_mtime >= pdf_path.stat().st_mtime

    if cache_fresh and not FORCE_RECONVERT:
        text = clean_markdown(md_path.read_text(encoding="utf-8"))  # clean old caches on read too
        print(f"Cached:     {pdf_path.name}  →  {md_path.name}")
    else:
        print(f"Converting: {pdf_path.name}")
        raw = md_converter.convert(str(pdf_path)).text_content
        text = clean_markdown(raw)  # strip margin-stamp / single-char-line noise
        md_path.write_text(text, encoding="utf-8")
        print(f"  → saved {md_path.name} ({len(text):,} chars, cleaned from {len(raw):,})")

    docs.append(Document(page_content=text, metadata={"source": pdf_path.name}))

print(f"\nLoaded {len(docs)} document(s) ({sum(len(d.page_content) for d in docs):,} chars total)")

### Step 2b — Chunk the Markdown

A full paper is too large to embed as a single vector, and retrieval works best on small, focused
passages. So each document is split into overlapping **chunks**.

**The logic of this stage:**
- **Measure length in tokens, not characters.** Chunk sizes are defined in the same token units the
  models use, so each chunk lines up with what the embedding model actually processes.
- **Aim for ~300-token chunks.** Small enough to be a precise unit of retrieval, large enough to
  carry a complete idea.
- **Overlap consecutive chunks (~50 tokens).** A fact that spans a boundary would otherwise be
  split across two chunks and lost; the overlap keeps it whole in at least one of them.
- **Split on natural boundaries.** Paragraphs and lines are preferred break points, so chunks stay
  readable rather than being cut mid-sentence.
- **Carry the source label along.** Every chunk keeps the metadata of the paper it came from.

**The trade-off to understand:** smaller chunks give more precise matches but less context per hit;
larger chunks give more context but noisier matches. The `300 / 50` setting is a solid
general-purpose starting point.

In [55]:
# ── Step 2b: Chunk into retrieval-friendly sizes ──────────────────────────────
from collections import Counter

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50,
)

doc_splits = splitter.split_documents(docs)

print(f"Total chunks: {len(doc_splits)}")
print(f"\nChunks per source:")
for source, count in Counter(d.metadata["source"] for d in doc_splits).items():
    print(f"  {source}: {count} chunks")

print(f"\nSample chunk (from {doc_splits[0].metadata['source']}):")
print(doc_splits[0].page_content[:300] + "...")

Total chunks: 406

Chunks per source:
  2005.11401v4 Retrieval-Augmented Generation.pdf: 91 chunks
  2308.08155v2 Autogen - Enabling Next-Gen LLM Apps via Multiagent Conversations.pdf: 196 chunks
  2309.06180v1-Efficient Memory Management for Large Language Model Serving with PagedAttention.pdf: 105 chunks
  2408.04619v1-TRANSFORMER EXPLAINER - Interactive Learning of Text-Generative Models.pdf: 14 chunks

Sample chunk (from 2005.11401v4 Retrieval-Augmented Generation.pdf):
Retrieval-Augmented Generation for
Knowledge-Intensive NLP Tasks

Patrick Lewis†‡, Ethan Perez(cid:63),

Aleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,

Mike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†

†Facebook AI Research; ‡...


## Step 3 — Store Embeddings in pgvector

The final ingestion stage turns each chunk into a vector (an **embedding**) and saves it in
pgvector so it can later be searched by meaning rather than keywords.

**PDF → MarkItDown → Chunks → Embeddings → pgvector**

**What an embedding is:** the model maps a chunk of text to a 1024-number vector where similar
meanings land close together. That geometric closeness is what makes "find the most relevant
passages" possible.

**The logic of this stage:**
- **Fingerprint the data.** A hash of all chunks is computed and stored alongside the collection,
  so it's always possible to tell whether what's in the database matches the current content.
- **Check what already exists.** Before embedding, the database is asked which documents are already
  stored. This is what makes the step **resumable** — finished papers are skipped on a rerun.
- **Embed one document at a time and save as you go.** Rather than one giant batch, each paper is
  embedded and written to pgvector before moving to the next, with progress printed along the way.
  If something fails partway through, the completed papers are already safely persisted.
- **Verify the result.** A quick similarity search confirms the store returns relevant passages.

**Why this design:**
- **Incremental & resumable** — embedding is the slow, costly step, so skipping already-finished
  work keeps reruns fast and protects against losing progress.
- **Persistent** — the vectors live in Postgres, so Part B reuses them with no re-embedding.
- **`FORCE_REEMBED`** wipes and rebuilds everything when you need a clean slate — for example after
  changing a PDF, since the skip check here works per document, not per chunk.

In [58]:
import hashlib
from collections import OrderedDict

import psycopg

COLLECTION_NAME = "langgraph_rag_demo"
FORCE_REEMBED = False  # set True to wipe + rebuild the whole collection


# ── Signature of the current chunks (stored for reference/change tracking) ─────
def compute_manifest_hash(splits) -> str:
    h = hashlib.sha256()
    h.update(str(len(splits)).encode())
    for d in splits:
        h.update(d.page_content.encode("utf-8"))
        h.update(d.metadata.get("source", "").encode("utf-8"))
    return h.hexdigest()


# ── Which source documents are already embedded in pgvector? ───────────────────
# langchain_postgres stores data in `langchain_pg_collection` (one row per
# collection) and `langchain_pg_embedding` (one row per vector, with cmetadata).

def read_embedded_sources(name: str) -> set[str]:
    """Return the set of `source` values already stored in the collection."""
    try:
        with psycopg.connect(pgdb_conn) as conn, conn.cursor() as cur:

            # Look up the collection's UUID in langchain_pg_collection by its name.
            # if not found, return an empty set (collection not yet created), else get the UUID to query langchain_pg_embedding for distinct sources.
            cur.execute("SELECT uuid FROM langchain_pg_collection WHERE name = %s", (name,))
            row = cur.fetchone()
            if row is None:
                return set()
            
            (coll_uuid,) = row

            # cmetadata->>'source' — the ->> operator extracts a value from a JSON/JSONB column as text. 
            # It pulls the source field out of the JSON metadata for each row and returns it as a plain string (e.g. "2005.11401v4 Retrieval-Augmented Generation.pdf").
            # The two arrows: ->> returns text, whereas -> would return a JSON value. Here we want a plain string to compare/collect in Python.
            cur.execute(
                "SELECT DISTINCT cmetadata->>'source' FROM langchain_pg_embedding "
                "WHERE collection_id = %s",
                (coll_uuid,),
            )

            # Return a set of the distinct source values, filtering out any None values (in case some rows have missing metadata).
            return {r[0] for r in cur.fetchall() if r[0] is not None}
    except psycopg.errors.UndefinedTable:
        return set()  # tables not created yet — first ever run


# ── Group chunks by their source document, preserving order ────────────────────
chunks_by_source: "OrderedDict[str, list]" = OrderedDict()
for d in doc_splits:
    # setdefault:   if the source key doesn't exist, create it with an empty list, then append the document to that list
    #               if it does exist, just append the document to the existing list
    chunks_by_source.setdefault(d.metadata.get("source", "unknown"), []).append(d)

manifest_hash = compute_manifest_hash(doc_splits)

# Connect to the collection (create if missing). Wipe first only on a full rebuild.
vectorstore = PGVector(
    embeddings=databricks_embeddings,
    collection_name=COLLECTION_NAME,
    connection=pgvectordb_conn,
    use_jsonb=True,
    pre_delete_collection=FORCE_REEMBED,
    collection_metadata={"manifest_hash": manifest_hash},
)

## ── Which source documents are already embedded in pgvector? ───────────────────
# Rebuild mode (`FORCE_REEMBED=True`): treat the collection as empty so all docs are embedded again.
# Normal mode: load distinct `source` values already stored in this pgvector collection to skip re-embedding.
already = set() if FORCE_REEMBED else read_embedded_sources(COLLECTION_NAME)

# ── Embed one document at a time, saving (persisting) to pgvector after each ────
total = len(chunks_by_source)
embedded, skipped = 0, 0

# Loop through each source document and its chunks, embedding them if not already present in the collection.
# 1-based indexing for user-friendly progress display.
for i, (source, chunks) in enumerate(chunks_by_source.items(), 1):
    if source in already:
        print(f"[{i}/{total}] skip      {source}  ({len(chunks)} chunks already embedded)")
        skipped += 1
        continue
    print(f"[{i}/{total}] embedding {source}  ({len(chunks)} chunks)…", flush=True)
    vectorstore.add_documents(chunks)  # embeds + writes to pgvector immediately
    print(f"[{i}/{total}] done      {source}  — saved {len(chunks)} chunks")
    embedded += 1

print(f"\nIngestion complete: {embedded} document(s) embedded, {skipped} already present.")

# Quick test — similarity search
test_results = vectorstore.similarity_search_with_score("What is RAG?", k=2)
for doc, score in test_results:
    print(f"score={score:.4f} | {doc.page_content[:100]}...")

[1/4] skip      2005.11401v4 Retrieval-Augmented Generation.pdf  (91 chunks already embedded)
[2/4] skip      2308.08155v2 Autogen - Enabling Next-Gen LLM Apps via Multiagent Conversations.pdf  (196 chunks already embedded)
[3/4] skip      2309.06180v1-Efficient Memory Management for Large Language Model Serving with PagedAttention.pdf  (105 chunks already embedded)
[4/4] skip      2408.04619v1-TRANSFORMER EXPLAINER - Interactive Learning of Text-Generative Models.pdf  (14 chunks already embedded)

Ingestion complete: 0 document(s) embedded, 4 already present.
score=0.3209 | 2 Methods

We explore RAG models, which use the input sequence x to retrieve text documents z and us...
score=0.3237 | Retrieve-and-Edit approaches Our method shares some similarities with retrieve-and-edit style
approa...


## Step 3 (Alternative) — Incremental Indexing with LangChain's RecordManager

*Optional — in a real project you'd pick **one** of the two Step 3 approaches.* This shows the same
"embed & store" step using LangChain's built-in **indexing API** instead of the hand-rolled loop
above.

**How it differs from the manual loop:**
- A **`RecordManager`** keeps a bookkeeping table (in Postgres) holding a hash of every chunk it has
  written. On each run, `index()` computes the delta: new chunks are embedded, unchanged chunks are
  skipped, and changed or removed chunks are **deleted and replaced** — automatically.
- This closes the two gaps in the manual approach: it detects change **per chunk** (not per whole
  document), and it **cleans up stale vectors** so nothing orphaned is left behind when content changes.
  
**Notes:**
- Runs against a **separate collection** so it doesn't clash with the manual pipeline above
  (mixing both on one collection would create duplicate vectors).

In [ ]:
# De-duplicate identical chunks first. index() keys each chunk by a hash of its
# content + metadata, so two chunks with the same text and source collapse to one key.
seen = set()
unique_splits = []
for d in doc_splits:
    k = (d.page_content, d.metadata.get("source"))
    if k not in seen:
        seen.add(k)
        unique_splits.append(d)

print(f"{len(doc_splits)} chunks → {len(unique_splits)} unique "
      f"({len(doc_splits) - len(unique_splits)} exact duplicates removed)")

# IMPORTANT: with cleanup="incremental", index() runs cleanup *per batch*. If one
# source's chunks span more than one batch (default batch_size=100, and some papers
# have >100 chunks), an early batch's cleanup deletes that source's not-yet-processed
# chunks, which the next batch re-adds — causing num_added == num_deleted every run.
# Fix: use a batch_size large enough that no single source is split across batches.
# Passing the full length puts everything in one batch (embeddings are still batched
# internally by the embeddings client's own chunk_size).
result = index(
    unique_splits,
    record_manager,
    rm_vectorstore,
    cleanup="incremental",   # remove stale vectors whose source content changed
    source_id_key="source",  # group chunks by this metadata key for cleanup
    key_encoder="sha256",    # stronger than the default SHA-1 chunk fingerprint
    batch_size=len(unique_splits),  # keep each source's chunks in a single batch
)
print(result)  # {'num_added', 'num_updated', 'num_skipped', 'num_deleted'}

# After the first reconciling run, re-running should report everything skipped
# (num_added == num_deleted == 0).

  ### What `index()` does under the hood

  `index()` orchestrates synchronization between your current `doc_splits`, the `RecordManager`, and the target vector store:
```mermaid
flowchart TD
    A[doc_splits input] --> B[Compute chunk key<br/>content + metadata<br/>sha256]
    B --> C{Key in<br/>RecordManager?}

    C -- No --> D[Embed chunk]
    D --> E[Insert vector in rm_vectorstore]
    E --> F[Write key/state to RecordManager]
    F --> G[num_added +1]

    C -- Yes, unchanged --> H[Skip chunk]
    H --> I[num_skipped +1]

    C -- Yes, changed --> J[Re-embed + update vector]
    J --> K[Update RecordManager entry]
    K --> L[num_updated +1]

    M[cleanup='incremental'<br/>source_id_key='source'] --> N[Find stale keys/vectors<br/>for updated/removed sources]
    N --> O[Delete stale vectors + records]
    O --> P[num_deleted +N]

    G --> Q[Return stats dict]
    I --> Q
    L --> Q
    P --> Q
```

  1. **Fingerprint each chunk**  
    It computes a stable key for each chunk (here with `key_encoder="sha256"`), based on content + metadata.

  2. **Check prior state in `RecordManager`**  
    It looks up which chunk keys were already indexed in the SQL bookkeeping table.

  3. **Apply the delta**
    - **New key** → embed + insert into `rm_vectorstore`
    - **Existing unchanged key** → skip
    - **Changed/removed key** (with `cleanup="incremental"`) → delete stale vector(s) and record(s)

  4. **Group cleanup by source**  
    With `source_id_key="source"`, cleanup is scoped by document source so stale chunks from updated files are removed safely.

  5. **Return run stats**  
    The result dict reports:
    - `num_added`
    - `num_updated`
    - `num_skipped`
    - `num_deleted`

  In short, `index()` keeps the vector store in sync with your latest chunks automatically, avoiding manual “what changed?” logic.

## Summary

### Ingestion Pipeline

| Stage | Technology | What it does |
|-------|-----------|--------------|
| 1. Convert | **MarkItDown** | PDF → clean Markdown (preserves headings, tables, structure) |
| 2. Chunk | `RecursiveCharacterTextSplitter` | Split into 300-token pieces with 50-token overlap |
| 3. Embed | `databricks-gte-large-en` (1024-dim) | Convert text chunks to dense vectors |
| 4. Store | **pgvector** (`PGVector`) | Persistent ANN similarity search in Postgres |

### Key Takeaways

- **MarkItDown** gives much cleaner text than raw PDF extraction — headings, tables, and
  structure are preserved, leading to better chunks and better retrieval
- The `.md` files and the pgvector collection are both **cached**, so re-running is incremental:
  conversion and embedding only happen for new or changed documents
- Embedding runs **one document at a time** and persists after each, so a failure part-way
  through doesn't lose completed work

### Next Step

The pgvector collection `langgraph_rag_demo` is now populated and persistent. Continue to
**`11b.langgraph_rag.ipynb`**, which connects to this same collection and builds a LangGraph
RAG agent on top of it — without re-running any of the ingestion above.